
# THE BRICK LEARNING ACADEMY


# 2 - Developing a Simple Pipeline

In this demonstration, we will create a simple Lakeflow Declarative Pipeline project using the new **ETL Pipeline multi-file editor** with declarative SQL.


### Learning Objectives

By the end of this lesson, you will be able to:
- Describe the SQL syntax used to create a Lakeflow Declarative Pipeline pipeline.
- Navigate the Lakeflow Declarative Pipeline ETL Pipeline multi-file editor to modify pipeline settings and ingest the raw data source file(s).
- Create, execute and monitor a Lakeflow Declarative Pipeline pipeline.

##  Developing and Running a Lakeflow Declarative  Pipeline with the ETL Pipeline Multi-File Editor

This course includes a simple, pre-configured Lakeflow Declarative Pipeline to explore and modify. In this section, we will:

- Explore the ETL Pipeline multi-file editor and the declarative SQL syntax  
- Modify pipeline settings  
- Run the Lakeflow Declarative Pipeline and explore the streaming tables and materialized view.

In this course we have starter files for you to use in your pipeline. This demonstration uses the folder **2 - Developing a Simple Pipeline Project**. To create a pipeline and add existing assets to associate it with code files already available in your Workspace (including Git folders) complete the following:

   a. In the left navigation bar, select the **Folder** ![Folder Icon](./Includes/images/folder_icon.png) icon to open the Workspace navigation.

   b. Navigate to the **Build Data Pipelines with Lakeflow Declarative Pipelines** folder (you may already be there).

   c. **(PLEASE READ)** For ease of use, open this same notebook in a separate tab:

    - Right-click the notebook in the left navigation.

    - Select **Open in a New Tab**.

   d. In the new tab, click the **three-dot (ellipsis) icon** ![Ellipsis Icon](./Includes/images/ellipsis_icon.png) in the folder navigation bar.

   e. Select **Create** → **ETL Pipeline**.

   f. Complete the pipeline creation page with the following:

    - **Name**: `Name-your-pipeline-using-this-notebook-name-add-your-first-name` 
    - **Default catalog**: Select your **labuser** catalog  
    - **Default schema**: Select your **default** schema (database)

   g. Select **Add existing assets**. In the popup, complete the following:

    - **Pipeline root folder**: Select the **2 - Developing a Simple DLT Pipeline Project** folder (`/Workspace/Users/your-lab-user-name/build-data-pipelines-with-dlt-3.0.0/Build Data Pipelines with DLT/2 - Developing a Simple DLT Pipeline Project`)

    - **Source code paths**: Within the same root folder as above, select the **orders** folder (`/Workspace/Users/your-lab-user-name/build-data-pipelines-with-dlt-3.0.0/Build Data Pipelines with DLT/2 - Developing a Simple DLT Pipeline Project/orders`)

   h. This will create a pipeline and associate the correct files for this demonstration.

**Example**

![Example Demo 2](./Includes/images/demo02_existing_assets.png)

3. In the new window, select the **orders_pipeline.sql** file and follow the instructions in the file. Leave this notebook as you will use it later.

![Orders File Directions](./Includes/images/demo02_select_orders_sql_file.png)

In [0]:

select * from thebricklearning.aianalyticsengineer.orders_bronze_demo2

In [0]:

select * from  thebricklearning.aianalyticsengineer.orders_silver_demo2

In [0]:

select * from thebricklearning.aianalyticsengineer.gold_orders_by_date_demo2

## C. Add a New File to Cloud Storage

1. After exploring and executing the pipeline by following the instructions in the **`orders_pipeline.sql`** file, run the cell below to add a new JSON file (**01.json**) to your volume at:  `/Volumes/dbacademy/ops/labuser-your-id/orders`.

   **NOTE:** If you receive the error `name 'DA' is not defined`, you will need to rerun the classroom setup script at the top of this notebook to create the `DA` object. This is required to correctly reference the path and successfully copy the file.

In [0]:
%python
# Define source and destination directories
source_dir = "/Volumes/thebricklearning/aianalyticsengineer/datasets/retailpipeline/orders_stream/"
destination_dir = "/Volumes/thebricklearning/aianalyticsengineer/datasets/orders/"

# List files in the source directory
files = dbutils.fs.ls(source_dir)

# Iterate through the files and copy each one
for file in files:
    source_file_path = file.path
    destination_file_path = destination_dir + file.name  # Destination path with the same file name
    
    # Copy the file to the destination
    dbutils.fs.cp(source_file_path, destination_file_path)
    print(f"Copied {file.name} to {destination_file_path}")


2. Complete the following steps to view the new file in your volume:

   a. Select the **Catalog** icon ![Catalog Icon](./Includes/images/catalog_icon.png) from the left navigation pane.  
   
   b. Expand your **dbacademy.ops.labuser** volume.  
   
   c. Expand the **orders** directory. You should see two files in your volume: **00.json** and **01.json**.

3. Run the cell below to view the data in the new **/orders/01.json** file. Notice the following:

   - The **01.json** file contains new orders.  
   - The **01.json** file has 25 rows.


In [0]:
%python
spark.sql(f'''
  SELECT *
  FROM json.`/Volumes/thebricklearning/aianalyticsengineer/datasets/orders/`
''').display()

4. Go back to the **orders_pipeline.sql** file and select **Run Pipeline** to execute your ETL pipeline again with the new file (Step 13).  

   Watch the pipeline run and notice that only 25 rows are added to the bronze and silver tables. 
   
   This happens because the pipeline has already processed the first **00.json** file (174 rows), and it is now only reading the new **01.json** file (25 rows), appending the rows to the streaming tables, and recomputing the materialized view with the latest data.

## D. Exploring Your Streaming Tables

1. View the new streaming tables and materialized view in your catalog. Complete the following:

   a. Select the catalog icon ![Catalog Icon](./Includes/images/catalog_icon.png) in the left navigation pane.

   b. Expand your **labuser** catalog.

   c. Expand the schemas **1_bronze_db**, **2_silver_db**, and **3_gold_db**. Notice that the two streaming tables and materialized view are correctly placed in your schemas.

      - **labuser.1_bronze_db.orders_bronze_demo2**

      - **labuser.2_silver_db.orders_silver_demo2**

      - **labuser.3_gold_db.orders_by_date_gold_demo2**

2. Run the cell below to view the data in the **labuser.1_bronze_db.orders_bronze_demo2** table. Before you run the cell, how many rows should this streaming table have?

   Notice that the following:
      - The table contains 199 rows (**00.json** had 174 rows, and **01.json** had 25 rows).
      - In the **source_file** column you can see the exact file the rows were ingested from.
      - In the **processing_time** column you can see the exact time the rows were ingested.

In [0]:
select * from thebricklearning.aianalyticsengineer.orders_bronze_demo2

In [0]:
select * from thebricklearning.aianalyticsengineer.orders_silver_demo2

In [0]:
select * from thebricklearning.aianalyticsengineer.gold_orders_by_date_demo2

3. Complete the following steps to view the history of the **orders_bronze_demo2** streaming table:

   a. Select the **Catalog** icon ![Catalog Icon](./Includes/images/catalog_icon.png) in the left navigation pane.  
   
   b. Expand the **labuser.01_bronze_db** schema.  
   
   c. Click the three-dot (ellipsis) icon next to the **orders_bronze_demo2** table.  
   
   d. Select **Open in Catalog Explorer**.  
   
   e. In the Catalog Explorer, select the **History** tab. Notice an error is returned because viewing the history of a streaming table requires **SHARED_COMPUTE**. In our labs we use a **DEDICATED (formerly single user)** cluster.

   f. Above your catalogs on the left select your compute cluster and change it to the provided **shared_warehouse**.

   ![Change Compute](./Includes/images/change_compute.png)  
   
   g. Go back and look at the last two versions of the table. Notice the following:  
   
      - In the **Operation** column, the last two updates were **STREAMING UPDATE**.  
      
      - Expand the **Operation Parameters** values for the last two updates. Notice both use `"outputMode": "Append"`.  
      
      - Find the **Operation Metrics** column. Expand the values for the last two updates. Observe the following:
      
         - It displays various metrics for the streaming update: **numRemovedFiles, numOutputRows, numOutputBytes, and numAddedFiles**.  
         
         - In the `numOutputRows` values, 174 rows were added in the first update, and 25 rows in the second.
   
   h. Close the Catalog Explorer.

## E. Viewing Lakeflow Declarative Pipelines with the Pipelines UI

After exploring and creating your pipeline using the **orders_pipeline.sql** file in the steps above, you can view the pipeline(s) you created in your workspace via the **Pipelines** UI.

1. Complete the following steps to view the pipeline you created:

   a. In the main applications navigation pane on the far left (you may need to expand it by selecting the ![Expand Navigation Pane](./Includes/images/expand_main_navigation.png) icon at the top left of your workspace) right-click on **Pipelines** and select **Open Link in a New Tab**.

   b. This should take you to the pipelines you have created. You should see your **2 - Developing a Simple Pipeline Project - labuser** pipeline.

   c. Select your **2 - Developing a Simple Pipeline Project - labuser**. Here, you can use the UI to modify the pipeline.

   d. Select the **Settings** button at the top. This will take you to the settings within the UI.

   e. Select **Schedule** to schedule the pipeline. Select **Cancel**, we will learn how to schedule the pipeline later.

   f. Under your pipeline name, select the drop-down with the time date stamp. Here you can view the **Pipeline graph** and other metrics for each run of the pipeline.

   g. Close the pipeline UI tab you opened.

## Additional Resources

- [Lakeflow Declarative Pipelines](https://docs.databricks.com/aws/en/dlt/) documentation.